# Week 5, Day 2: Pydantic AI

## What this lab covers

This lab configures a Pydantic AI agent with an Azure OpenAI model and runs a first asynchronous request. It adds typed tools for a shared SQLite task board, connects a local filesystem server through an MCP toolset, and creates a worker that plans and completes a translation task using both capabilities.

### Day 02: Pydantic AI

In [1]:
# Necessary library imports
import os
import subprocess
from pathlib import Path
from openai import AsyncOpenAI
from dotenv import load_dotenv
from pydantic_ai import Agent
from openai import AsyncAzureOpenAI 
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.mcp import MCPToolset
from fastmcp.client.transports import StdioTransport

from IPython.display import display, Markdown
load_dotenv(override=True)

True

In [2]:
# Let's see if the API key is working/helping us to call LLM from Azure Foundry
# From OpenAI
AZURE_OPENAI_API_KEY= os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_MODEL_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_DEPLOYMENT_GPT_41 = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_41")
AZURE_OPENAI_DEPLOYMENT_GPT_54_mini = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_54_mini")
AZURE_OPENAI_DEPLOYMENT_GPT_55 = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_55")
AZURE_OPENAI_DEPLOYMENT_GPT_4O_mini = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_4O_mini")
if AZURE_OPENAI_API_KEY:
    print("AZURE_OPENAI_API_KEY is available")
else:
    print("AZURE_OPENAI_API_KEY is not available")


AZURE_OPENAI_API_KEY is available


#### Step 01: Create an agent

In [3]:
# let's prepare a client of openai-azure foundry
openai_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)

In [4]:
# let's create a model and agent
model = OpenAIChatModel(
    model_name=AZURE_OPENAI_DEPLOYMENT_GPT_54_mini,
    provider=OpenAIProvider(openai_client=openai_client)
)

agent = Agent(
    model=model,
    system_prompt = "You are a concise, friendly assitant. Reply in a single short sentence."
)

#### Step 02: Run the agent

In [5]:
result =  await agent.run("Say hello in Spanish.")
print(result.output)

Hola.


#### A project this week: a SQLite todo board

In [6]:
import board
board.reset_board()
board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to Spanish.txt.")
board.list_todos()

[{'id': 1,
  'parent_id': None,
  'task': 'Read notes.txt, translate its contents into natural Spanish, and write the Spanish to Spanish.txt.',
  'status': 'pending',
  'result': ''}]

In [7]:
board.show_board()

Goal #1: Read notes.txt, translate its contents into natural Spanish, and write the Spanish to Spanish.txt.

#### Step 03: Add tools

In [15]:
# let's define some functions for interacting with the board
from pydantic_ai import RunContext

@agent.tool_plain
def show_todos() -> list[dict]:
    """List every todo on board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

@agent.tool_plain
def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions.
    
    Args: 
        goal_id: The id of the goal to break down.
        steps: Short descriptions of the steps ot take, in order.    
    """
    return {"goal_id": goal_id, "steps_id": [board.add_step(goal_id, step) for step in steps]}

@agent.tool_plain
def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a steps or the goal) with this id as done and record a short result summary.
    
    Args: 
        task_id: The id of the todo to mark it done.
        result: a short summary of what was accomplished.
    """
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "result": result, "status": "done"}


In [9]:
complete_task

<function __main__.complete_task(task_id: int, result: str) -> dict>

In [10]:
# make a function to print the results of agent call in Markdown format
def print_markdown(result: dict) -> str:
    print("Agent's/LLM's answer is: \n")
    return display(Markdown(result.output))


In [11]:
# let's create an agent
board_agent = Agent(
    model=model,
    system_prompt="You help manage a shared todo board.",
    tools=[show_todos,complete_task]
)

# let's invoke it
result = await board_agent.run("What is on board right now, and what is its status..?")
print_markdown(result)

Agent's/LLM's answer is: 



There is 1 item on the board right now:

- **ID 1** — **Read notes.txt, translate its contents into natural Spanish, and write the Spanish to Spanish.txt.**
  - **Status:** pending



#### Step 04: Add a MCP(Model Context Protocol)

In [12]:
# import some required libraries for filesystem mcp server 
from pydantic_ai.mcp import MCPToolset

filesystem_toolset = MCPToolset(
    "http://127.0.0.1:8000/sse"
)

In [13]:
# let's call the agent with above custom filesystem mcp server as tool
agent = Agent(
    model=model,
    toolsets = [filesystem_toolset],
    system_prompt="You can read and write files in your workspace. Use your tools to do what is asked."
)
result = await agent.run("Read notes.txt summarize it in a one short sentence.")
print_markdown(result)

Agent's/LLM's answer is: 



It says the team is building a small language tutor step by step, with each helper completing one task carefully.

#### Step 05: Let's put agent in a loop with a goal

In [17]:
# let's define INSTRUCTIONS, agent with it's tools of handling filesystem and invoke it

INSTRUCTIONS="""
You're a careful worker with a todo board and a set of file tools.
Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into,
added to the board under the goal. Then carry them out with your file tools, marking each step done as you finish it. Once the steps are all done, close the goal.
Your files lives in the single folder, your tools are allowed to use.
"""
worker = Agent(
    model=model,
    system_prompt=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task],
    toolsets=[filesystem_toolset]
)
board.reset_board()
goal_id = board.add_goal("Read notes.txt, translate its contents into natural spanish, and write the spanish to spanish.txt")

result=await worker.run("Please work on pending goal on the board")
print_markdown(result)
board.show_board()

Agent's/LLM's answer is: 



Done — I translated `notes.txt` into natural Spanish and saved it to `spanish.txt`.

Goal #1: Read notes.txt, translate its contents into natural spanish, and write the spanish to spanish.txt  Completed the translation task and saved the Spanish version in spanish.txt.
  Step #2: Inspect notes.txt and understand the source text.  Inspected notes.txt and understood the source text.
  Step #3: Translate the text into natural Spanish.  Translated the notes into natural Spanish.
  Step #4: Write the Spanish translation to spanish.txt.  Wrote the Spanish translation to spanish.txt.
  Step #5: Verify the output file and complete the goal.  Verified spanish.txt was created and contains the translated text.